# The Witness Stand — SFT + GRPO Training Notebook

Run this notebook in **Google Colab with GPU enabled**.

Recommended flow:
1. Install dependencies
2. Clone repo
3. Add API keys
4. Build dossier
5. Run health checks
6. Run smoke training
7. Run standard training
8. Save / push adapter

Do **not** start with standard/full training. Always run smoke first.


## 1. Check GPU

In [ ]:
!nvidia-smi

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Sat Apr 25 13:50:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install Unsloth + TRL dependencies

In [ ]:
%%capture
!pip uninstall -y torch torchvision torchaudio xformers unsloth unsloth_zoo
!pip install --upgrade pip

# Install matching PyTorch + TorchVision CUDA wheels
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Install Unsloth after torch/torchvision are clean
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# Install training stack
!pip install -U trl transformers accelerate peft bitsandbytes datasets python-dotenv groq pyyaml requests fastapi uvicorn


## 3. Clone repository

In [ ]:
%cd /content
!rm -rf witness-stand
!git clone https://github.com/Rohitchandramouli/witness-stand.git
%cd /content/witness-stand
!pip install -e . --quiet
!pwd
!ls


## 4. Add API keys

Paste your keys only inside Colab runtime. Do not commit them to GitHub.

Required:
- `GROQ_API_KEY` for dossier/persona generation if needed
- `HF_TOKEN` only if pushing adapter to Hugging Face


In [ ]:
import os
from getpass import getpass

if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter GROQ_API_KEY: ")

if not os.getenv("HF_TOKEN"):
    hf = getpass("Enter HF_TOKEN, or press Enter to skip: ")
    if hf.strip():
        os.environ["HF_TOKEN"] = hf.strip()

print("GROQ key set:", bool(os.getenv("GROQ_API_KEY")))
print("HF token set:", bool(os.getenv("HF_TOKEN")))


## 5. Verify project files

In [ ]:
!python -m py_compile scripts/train_witness_stand.py
!python -m py_compile environment.py agent/prompt.py agent/parser.py agent/heuristics.py


## 6. Build dossier

Run this once per fresh Colab runtime.


In [ ]:
!python scripts/00_build_dossier.py
!python scripts/01_inspect_dossier.py


## 7. Run health checks

In [ ]:
!python scripts/08_health_all.py


## 8. Smoke training

This is mandatory. It checks that:
- model loads
- SFT runs
- GRPO reward works
- reconstruction reward path works
- adapter saves

Do not skip this.


In [ ]:
!python scripts/train_witness_stand.py \
  --mode smoke \
  --output_dir /content/witness-stand-checkpoints-smoke


## 9. Inspect smoke training log

Check that baseline/post-training are printed and no reward errors dominate.


In [ ]:
!ls -lah logs/training || true
!cat logs/training/latest_training.json | head -120 || true


## 10. Standard training

Run this only after smoke succeeds.

This is the main hackathon training run.


In [ ]:
!python scripts/train_witness_stand.py \
  --mode standard \
  --output_dir /content/witness-stand-checkpoints-standard


## 11. Optional full training

Run only if standard training is stable and you have enough GPU time/credits.


In [ ]:
# Optional. Uncomment only if needed.
# !python scripts/train_witness_stand.py \
#   --mode full \
#   --output_dir /content/witness-stand-checkpoints-full


## 12. Push adapter to Hugging Face Hub

Run only after you are happy with the result.


In [ ]:
# Optional final push.
# Make sure HF_TOKEN is set and hf_repo inside TrainConfig is correct.
# !python scripts/train_witness_stand.py \
#   --mode standard \
#   --output_dir /content/witness-stand-checkpoints-final \
#   --push_to_hub


## 13. Download checkpoints/logs from Colab

In [ ]:
!zip -r /content/witness_stand_training_outputs.zip /content/witness-stand-checkpoints-smoke /content/witness-stand-checkpoints-standard logs/training || true
from google.colab import files
files.download("/content/witness_stand_training_outputs.zip")


## Notes

Recommended team strategy:
- Member 1: run smoke + standard
- Member 2: run standard with more GRPO steps
- Member 3: keep credits for final Space/demo or backup run

Example alternate run:

```bash
python scripts/train_witness_stand.py --mode standard --grpo_steps 80 --eval_episodes 5
```
